# Add NYC Weather to Silver

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    TimestampType,
    DecimalType
)

In [0]:
from pyspark import pipelines as dp

## Variables

## Schema Definition

In [0]:
schema = StructType(
    [
        StructField(
            name="datetime",
            dataType=TimestampType(),
            nullable=False,
            metadata={"comment": "Timestamp of weather observation (hourly)"}
        ),
        StructField(
            name="temperature",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "Temperature at 2 meters above ground in °C"}
        ),
        StructField(
            name="humidity",
            dataType=IntegerType(),
            nullable=True,
            metadata={"comment": "Relative humidity at 2 meters above ground in %"}
        ),
        StructField(
            name="apparent_temperature",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "Apparent temperature (feels like) in °C"}
        ),
        StructField(
            name="precipitation",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "Total precipitation in mm"}
        ),
        StructField(
            name="wind_speed",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "Wind speed at 10 meters above ground in km/h"}
        ),
    ]
)

## ETL

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="analytics.silver.weather_stm_weather",
    # Beschreibung der Tabelle
    comment="This table shows the different weather in nyc at each hour in 2025",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    schema=schema,
)
def weather_stm_weather():
    df = spark.read.table("analytics.bronze.dwh_nyc_weather")
    df = df.withColumnsRenamed(
        {
            "observation_time": "datetime",
            "temperature_2m": "temperature",
            "relative_humidity_2m": "humidity",
            "apparent_temperature": "apparent_temperature",
            "precipitation": "precipitation",
            "wind_speed_10m": "wind_speed",
        }
    )
    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
    
    return df